<a href="https://colab.research.google.com/github/arelkeselbri/pgc305/blob/main/dpo_treinamento_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treinando LLMs com DPO (Direct Preference Optimization)
### Aula passo a passo

Neste notebook vamos aprender, na prática, como alinhar um modelo de linguagem (LLM) às preferências humanas usando **DPO (Direct Preference Optimization)**.

**O que você vai aprender:**
1. A intuição e a matemática por trás do DPO
2. Como preparar um dataset de preferências (`prompt`, `chosen`, `rejected`)
3. Como carregar um modelo base pequeno (para rodar em CPU/GPU modesta)
4. Como configurar e rodar o `DPOTrainer` da biblioteca `trl`
5. Como comparar as respostas do modelo antes e depois do treino
6. Como salvar o modelo treinado

> ⚠️ Para fins didáticos, usamos um modelo bem pequeno (`Qwen/Qwen2.5-0.5B-Instruct` ou `gpt2`) e um subconjunto reduzido do dataset. Em produção, você usaria um modelo maior e o dataset completo, idealmente com GPU.


In [ ]:
!pip install trl

## 1. O que é DPO e por que ele existe?

Depois do pré-treino e do fine-tuning supervisionado (SFT), queremos que o modelo **prefira** certas respostas a outras (mais úteis, mais seguras, mais educadas etc.). O método clássico para isso é o **RLHF** (Reinforcement Learning from Human Feedback), que:

1. Treina um *reward model* a partir de comparações humanas (resposta A vs resposta B);
2. Usa PPO (um algoritmo de RL) para otimizar o LLM contra esse reward model.

Isso é **complexo, instável e caro** (precisa treinar 2 modelos, fazer rollout, etc.).

O **DPO** (artigo: *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*, Rafailov et al., 2023) mostra que é possível **pular o reward model e o RL** e otimizar diretamente o LLM com uma função de perda simples, usando apenas pares `(prompt, resposta_preferida, resposta_rejeitada)`.

### A ideia central

Dado um par de respostas para o mesmo prompt — uma **escolhida** ($y_w$, *chosen*) e uma **rejeitada** ($y_l$, *rejected*) — o DPO ajusta o modelo de política $\pi_\theta$ para aumentar a probabilidade relativa da resposta escolhida, mas **sem deixar o modelo se afastar demais** de um modelo de referência $\pi_{ref}$ (geralmente o próprio modelo antes do DPO, congelado).

A perda do DPO é:

$$
\mathcal{L}_{DPO}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]
$$

Onde:
- $\sigma$ é a função sigmoide;
- $\beta$ controla o quão longe o modelo pode se afastar da referência (um "freio" de regularização);
- O termo dentro do $\log\sigma$ é, na prática, "o quanto o modelo aumentou a chance da resposta boa, relativo à chance da resposta ruim, comparado com o modelo de referência".

Na prática isso significa: **não precisamos de reward model nem de RL** — apenas duas passagens *forward* (modelo atual e modelo de referência) e uma função de perda parecida com uma classificação binária.


## 2. Preparando o ambiente

Vamos instalar as bibliotecas necessárias:

- `transformers`: para carregar modelo e tokenizer
- `trl`: contém o `DPOTrainer`, já implementado e otimizado
- `datasets`: para carregar e manipular o dataset de preferências
- `accelerate`: para gerenciar dispositivos (CPU/GPU) de forma transparente
- `peft` (opcional): para treinar com LoRA, reduzindo drasticamente o uso de memória


In [ ]:
# Descomente para instalar (recomendado rodar em um ambiente com GPU)
# !pip install -q transformers trl datasets accelerate peft bitsandbytes


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")


## 3. Carregando o modelo base

Vamos usar um modelo pequeno para que este notebook seja executável até em uma GPU modesta (ou CPU, com paciência). A ideia é exatamente a mesma para modelos maiores (Llama, Mistral, Qwen 7B etc.) — só trocaria o `model_name` e, idealmente, usaria LoRA + quantização.

Importante: o DPO normalmente é aplicado **depois** de um modelo já ter passado por SFT (Supervised Fine-Tuning). Aqui, para simplificar a aula, partimos direto de um modelo *instruct* pré-treinado, que já sabe seguir instruções básicas.


In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # modelo pequeno (~0.5B params), ideal para estudo

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).to(device)

print(model.config.model_type, "carregado com sucesso.")


## 4. O dataset de preferências

O `DPOTrainer` espera um dataset com (no mínimo) três colunas:

| coluna     | significado                                  |
|------------|-----------------------------------------------|
| `prompt`   | a pergunta/instrução enviada ao modelo         |
| `chosen`   | a resposta **preferida** (melhor)              |
| `rejected` | a resposta **rejeitada** (pior)                |

Vamos usar um dataset público já formatado nesse estilo: `trl-lib/ultrafeedback_binarized` (uma versão tratada do UltraFeedback, com pares de respostas vencedora/perdedora). Para a aula, pegamos só uma fatia pequena.


In [ ]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

# Para fins didáticos, usamos só uma fração pequena do dataset
dataset = dataset.shuffle(seed=42).select(range(200))

print(dataset)
print("\nExemplo:")
exemplo = dataset[0]
print("PROMPT:", exemplo["chosen"][0]["content"][:300])
print("\nCHOSEN RESPONSE:", exemplo["chosen"][1]["content"])
print("\nREJECTED RESPONSE:", exemplo["rejected"][1]["content"])

### Sobre o formato `chosen` / `rejected`

Dependendo do dataset, `chosen` e `rejected` podem vir como:
- strings simples (texto puro da resposta), ou
- listas de mensagens no formato de chat (`[{"role": "user", ...}, {"role": "assistant", ...}]`)

O `DPOTrainer` moderno da `trl` lida com os dois formatos automaticamente, desde que o `tokenizer` tenha um `chat_template` configurado (o que já vem pronto em modelos *Instruct* como o que carregamos). Se você usar um dataset/modelo diferente, vale inspecionar uma amostra como fizemos acima antes de treinar.


## 5. Testando o modelo *antes* do DPO

Antes de treinar, vamos gerar uma resposta com o modelo original, para depois comparar com o resultado pós-DPO.


In [ ]:
def gerar_resposta(modelo, prompt_texto, max_new_tokens=150):
    mensagens = [{"role": "user", "content": prompt_texto}]
    entrada = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(entrada, return_tensors="pt").to(device)

    with torch.no_grad():
        saida = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id,
        )
    texto = tokenizer.decode(saida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return texto

prompt_teste = "Explique o que é energia escura em poucas frases."
print("=== Resposta ANTES do DPO ===")
print(gerar_resposta(model, prompt_teste))


## 6. Configurando o `DPOConfig`

Os hiperparâmetros mais importantes:

- **`beta`**: o quão rígido é o "freio" contra o modelo de referência. Valores comuns: `0.1` a `0.5`. Beta alto = mudanças mais conservadoras; beta baixo = o modelo pode se afastar mais da referência.
- **`learning_rate`**: geralmente bem menor que no SFT (ex: `5e-6` a `5e-5`).
- **`per_device_train_batch_size`**: cada exemplo de DPO consome o dobro de memória de um exemplo normal (porque calcula log-probs para `chosen` e `rejected`, no modelo atual e no modelo de referência).
- **`num_train_epochs`**: poucas épocas já costumam ser suficientes (1–3) para evitar *overfitting* nas preferências.


In [ ]:
training_args = DPOConfig(
    output_dir="./dpo-qwen-0.5b-demo",
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=torch.cuda.is_available(),
)


## 7. Criando o `DPOTrainer`

Note que **não precisamos passar um `ref_model` explicitamente**: se deixarmos `None`, a `trl` cria automaticamente uma cópia congelada do modelo atual para servir de referência (ou, se estivermos usando LoRA, reutiliza o modelo base sem os adaptadores).


In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # a trl cria a referência automaticamente
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)


## 8. Treinando

Agora vamos rodar o treinamento. Acompanhe a métrica `loss` (deve cair) e, se disponível, `rewards/accuracies` (a fração de pares em que o modelo já prefere corretamente a resposta `chosen` — deve subir em direção a 1.0).


In [ ]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 9. Testando o modelo *depois* do DPO

Vamos gerar uma resposta para o mesmo prompt de antes e comparar qualitativamente.


In [ ]:
print("=== Resposta DEPOIS do DPO ===")
print(gerar_resposta(trainer.model, prompt_teste))


## 10. Salvando o modelo treinado

Por fim, salvamos o modelo e o tokenizer para uso posterior (inferência, upload ao Hugging Face Hub, etc.).


In [ ]:
trainer.save_model("./dpo-qwen-0.5b-demo/final")
tokenizer.save_pretrained("./dpo-qwen-0.5b-demo/final")
print("Modelo salvo em ./dpo-qwen-0.5b-demo/final")


## 11. Resumo e próximos passos

Você acabou de treinar um LLM com DPO! Recapitulando o fluxo:

1. Partimos de um modelo já com SFT (`Qwen2.5-0.5B-Instruct`);
2. Carregamos um dataset de pares `(prompt, chosen, rejected)`;
3. Configuramos o `DPOConfig` (destacando o hiperparâmetro `beta`);
4. Treinamos com `DPOTrainer`, que internamente calcula a perda de preferência comparando o modelo atual com uma referência congelada;
5. Comparamos as respostas antes/depois e salvamos o modelo.

**Para ir além:**
- Treinar com **LoRA** (via `peft`) para reduzir uso de memória em modelos maiores;
- Avaliar com métricas mais robustas (ex: um *judge* LLM, ou um conjunto de avaliação humano);
- Explorar variantes como **IPO**, **KTO** ou **ORPO**, também implementadas na `trl`, que resolvem alguns problemas teóricos do DPO original;
- Ler o paper original: *Rafailov et al., 2023 — "Direct Preference Optimization: Your Language Model is Secretly a Reward Model"* (arXiv:2305.18290).

### Referências
- Documentação da TRL: https://huggingface.co/docs/trl
- Paper do DPO: https://arxiv.org/abs/2305.18290
- Dataset usado: https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized
